# Filter

Jonas Frei, 15.10.2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control

## Problematik am Beispiel eines Encoders

Erzeugen eines Encodersignals

In [ ]:
dt = 0.0001
t = np.arange(0, 0.5 + dt/2, dt)
x = np.sin(9*2*np.pi*t)

Diskretisieren des Signals

In [ ]:
def discretize(x, bins):
    """Teilt x in `bins` gleich breite Intervalle über [min(x), max(x)]
    ein und gibt den 1-basierten Intervallindex jedes Werts zurück."""
    xmin, xmax = x.min(), x.max()
    idx = np.floor((x - xmin)/(xmax - xmin)*bins)
    idx = np.clip(idx, 0, bins - 1)
    return (idx + 1).astype(int)


bins = 500
x_disc = (discretize(x, bins) - 1)/(bins - 1)*2 - 1

Berechnen der Geschwindigkeit durch Ableiten

In [ ]:
xd = np.concatenate(([0], np.diff(x)/dt))
xd_disc = np.concatenate(([0], np.diff(x_disc)/dt))

Plotten

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(9, 7))

axs[0].plot(t, x_disc, linewidth=1.5, label="Diskretisiert")
axs[0].plot(t, x, linewidth=1.5, label="Echtes Signal")
axs[0].set_title("Mit Encoder gemessene Position")
axs[0].legend()
axs[0].set_xlabel("t [s]")
axs[0].set_ylabel("x [m]")

axs[1].plot(t, xd_disc, linewidth=1.5, label="Diskretisiert")
axs[1].plot(t, xd, linewidth=1.5, label="Echtes Signal")
axs[1].set_title("Daraus abgeleitete Geschwindigkeit")
axs[1].legend()
axs[1].set_xlabel("t [s]")
axs[1].set_ylabel("dx/dt [m/s]")

plt.tight_layout()
plt.show()

## Tiefpass, Hochpass und Notch Filter

In [ ]:
s = control.tf('s')

Tiefpassfilter

In [ ]:
K_tpf = 10
omega_tpf = 10000
G_tpf = K_tpf*omega_tpf/(s + omega_tpf)

Hochpassfilter

In [ ]:
K_hpf = 1
omega_hpf = 100
G_hpf = K_hpf*s/(omega_hpf + s)

Notch Filter

In [ ]:
K_notch = 0.1
sigma = 1000
omega_notch = 1000
G_notch = K_notch*(s**2 + omega_notch**2)/(s**2 + sigma*s + omega_notch**2)

Plotten

In [ ]:
control.bode_plot([G_tpf, G_hpf, G_notch], dB=True, Hz=True, deg=True,
                   label=['Tiefpassfilter', 'Hochpassfilter', 'Notch Filter'],
                   title='Bodediagramm')
plt.show()

## Fortsetzung Beispiel Encoder

Erstellen unterschiedlicher Tiefpassfilter bei 5000 Hz, 500 Hz, und 50 Hz

In [ ]:
G_tpf5000 = 5000/(s + 5000)
G_tpf500 = 500/(s + 500)
G_tpf50 = 50/(s + 50)

Filtern des diskretisierten Geschwindigkeitssignals

In [ ]:
xd_disc_filtered_5000 = control.forced_response(G_tpf5000, T=t, U=xd_disc).outputs
xd_disc_filtered_500 = control.forced_response(G_tpf500, T=t, U=xd_disc).outputs
xd_disc_filtered_50 = control.forced_response(G_tpf50, T=t, U=xd_disc).outputs

Plotten

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(t, xd_disc, linewidth=1.5, label="Diskretisiert")
ax.plot(t, xd_disc_filtered_5000, linewidth=1.5, label="Gefiltert @ 5000 Hz")
ax.plot(t, xd_disc_filtered_500, linewidth=1.5, label="Gefiltert @ 500 Hz")
ax.plot(t, xd_disc_filtered_50, linewidth=1.5, label="Gefiltert @ 50 Hz")
ax.plot(t, xd, linewidth=1.5, label="Wahres Signal")
ax.legend(loc="best")
ax.set_title("Gefilterte Geschwindigkeit (Tiefpassfilter)")
ax.set_xlabel("t [s]")
ax.set_ylabel("dx/dt [m/s]")
plt.show()